# Module 6: Artifacts -- Brain or Noise?

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup & Imports

This notebook covers **EEG artifacts** -- non-brain signals that contaminate recordings.

A lot of what shows up on the EEG screen is not brain activity at all. Eye blinks, muscle tension, heartbeats, loose electrodes, power lines -- they all contaminate the recording. An experienced neurologist can spot these instantly. By the end of this module, so can you.

**The Five Usual Suspects:**

| Artifact | Source | Key Feature | Worst Location |
|----------|--------|-------------|----------------|
| Eye blink | Corneoretinal dipole | Large, slow, positive | Fp1, Fp2 |
| Muscle (EMG) | Jaw, facial muscles | High-frequency fuzz | Temporal |
| 60 Hz line noise | Power lines | Perfect sinusoidal | All channels |
| ECG | Heart electrical field | Periodic sharp waves | Diffuse |
| Electrode pop | Loose electrode | Sudden spike, 1 channel | Single electrode |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
%matplotlib inline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Sampling parameters
fs = 500
duration = 3
t = np.arange(0, duration, 1/fs)

**Exercise 1:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 2: Clean EEG Baseline

First, we generate a clean EEG signal as our baseline. This represents what the brain signal looks like without contamination: a mix of alpha (10 Hz), theta (6 Hz), and some beta (18 Hz) activity.

In [ ]:
# Clean EEG baseline: alpha + theta + beta + noise
np.random.seed(42)
clean_eeg = (25 * np.sin(2*np.pi*10*t + 0.3) +
             10 * np.sin(2*np.pi*6*t + 1.2) +
             5 * np.sin(2*np.pi*18*t + 0.7) +
             8 * np.random.randn(len(t)))

plt.figure(figsize=(12, 3))
plt.plot(t, clean_eeg, 'b-', linewidth=1, alpha=0.8)
plt.xlabel('Time (s)')
plt.ylabel('uV')
plt.title('Clean EEG (no artifacts)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Exercise 2:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 3: Artifact 1: Eye Blink

**Source:** The eye acts as a dipole (cornea positive, retina negative). When you blink, the eyelid slides over this dipole, generating a large voltage change detected by frontal electrodes (Fp1/Fp2).

**Key features:** Large amplitude (~100-200 uV), slow (<2 Hz envelope), positive polarity, duration ~200-400 ms.

**How to fix:** Ask patient to fixate. Reject contaminated epochs. ICA subtraction.

In [ ]:
# Generate eye blink artifact
def generate_blink(t, blink_times=[1.0, 2.2], amplitude=180):
    """Simulate eye blink artifact."""
    blink = np.zeros(len(t))
    for bt in blink_times:
        dt = t - bt
        mask = (dt > -0.05) & (dt < 0.35)
        phase = (dt[mask] + 0.05) / 0.4
        blink[mask] = amplitude * np.sin(np.pi * phase) * np.exp(-2 * phase)
    return blink

blink_artifact = generate_blink(t)
contaminated_blink = clean_eeg + blink_artifact

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, clean_eeg, 'b-', linewidth=1)
axes[0].set_title('Clean EEG', fontsize=11)
axes[0].set_ylabel('uV')

axes[1].plot(t, blink_artifact, color='#e74c3c', linewidth=1.5)
axes[1].set_title('Eye Blink Artifact (alone)', fontsize=11, color='#e74c3c')
axes[1].set_ylabel('uV')

axes[2].plot(t, contaminated_blink, color='#c0392b', linewidth=1)
axes[2].set_title('Contaminated EEG (clean + blink)', fontsize=11, color='#c0392b')
axes[2].set_ylabel('uV')
axes[2].set_xlabel('Time (s)')

fig.suptitle('Eye Blink Artifact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Exercise 3:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 4: Artifact 2: Muscle (EMG)

**Source:** Contraction of scalp, jaw, or facial muscles near electrodes. Common triggers: jaw clenching, teeth grinding, frowning, talking.

**Key features:** High-frequency (>20 Hz), broadband 'fuzz,' often in bursts, predominantly over temporal electrodes (near temporalis muscle).

**How to fix:** Ask patient to relax. Low-pass filter (may lose real high-frequency data). ICA.

In [ ]:
# Generate muscle (EMG) artifact
def generate_muscle(t, start=0.8, end=1.8, fs=500):
    """Simulate muscle artifact -- high-frequency broadband burst."""
    np.random.seed(77)
    muscle = np.zeros(len(t))
    # Envelope: active during burst period
    envelope = np.zeros(len(t))
    mask = (t > start) & (t < end)
    envelope[mask] = np.sin(np.pi * (t[mask] - start) / (end - start))
    # Broadband high-frequency content
    for k in range(1, 13):
        freq = 30 + k * 15  # 45 Hz to 195 Hz
        muscle += np.sin(2*np.pi*freq*t + k*2.3 + np.random.rand()*6.28)
    muscle *= envelope * 4
    return muscle

muscle_artifact = generate_muscle(t)
contaminated_muscle = clean_eeg + muscle_artifact

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, clean_eeg, 'b-', linewidth=1)
axes[0].set_title('Clean EEG', fontsize=11)
axes[0].set_ylabel('uV')

axes[1].plot(t, muscle_artifact, color='#e67e22', linewidth=1)
axes[1].set_title('Muscle Artifact (alone)', fontsize=11, color='#e67e22')
axes[1].set_ylabel('uV')

axes[2].plot(t, contaminated_muscle, color='#d35400', linewidth=1)
axes[2].set_title('Contaminated EEG (clean + muscle)', fontsize=11, color='#d35400')
axes[2].set_ylabel('uV')
axes[2].set_xlabel('Time (s)')

fig.suptitle('Muscle (EMG) Artifact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Exercise 4:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 5: Artifact 3: 60 Hz Power Line Noise

**Source:** Electromagnetic interference from AC power lines (60 Hz in North America, 50 Hz in Europe). Affects all channels equally.

**Key features:** Perfect sinusoidal oscillation at exactly 60 Hz. Constant amplitude.

**How to fix:** Notch filter at 60 Hz. Check electrode impedances (high impedance makes it worse).

In [ ]:
# Generate 60 Hz line noise
line_noise = 15 * np.sin(2*np.pi*60*t)
contaminated_line = clean_eeg + line_noise

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, clean_eeg, 'b-', linewidth=1)
axes[0].set_title('Clean EEG', fontsize=11)
axes[0].set_ylabel('uV')

axes[1].plot(t, line_noise, color='#8e44ad', linewidth=1)
axes[1].set_title('60 Hz Line Noise (alone)', fontsize=11, color='#8e44ad')
axes[1].set_ylabel('uV')

axes[2].plot(t, contaminated_line, color='#6c3483', linewidth=1)
axes[2].set_title('Contaminated EEG (clean + 60 Hz)', fontsize=11, color='#6c3483')
axes[2].set_ylabel('uV')
axes[2].set_xlabel('Time (s)')

fig.suptitle('60 Hz Power Line Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Exercise 5:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 6: Artifact 4: ECG (Cardiac)

**Source:** The heart's electrical field is strong enough to be picked up by scalp electrodes. The sharp QRS complex of each heartbeat appears as periodic spikes.

**Key features:** Sharp periodic deflections at ~1 Hz (matching heart rate). Regular timing.

**How to fix:** ICA component removal. ECG reference subtraction.

In [ ]:
# Generate ECG artifact
def generate_ecg(t, heart_rate_hz=1.25):
    """Simulate ECG artifact at given heart rate (1.25 Hz = 75 bpm)."""
    ecg = np.zeros(len(t))
    phase = (t * heart_rate_hz) % 1.0
    # QRS-like spike
    qrs_mask = (phase > 0.28) & (phase < 0.35)
    ecg[qrs_mask] = 40 * np.sin(np.pi * (phase[qrs_mask] - 0.28) / 0.07)
    # T-wave-like
    t_mask = (phase > 0.42) & (phase < 0.55)
    ecg[t_mask] += 12 * np.sin(np.pi * (phase[t_mask] - 0.42) / 0.13)
    return ecg

ecg_artifact = generate_ecg(t)
contaminated_ecg = clean_eeg + ecg_artifact

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, clean_eeg, 'b-', linewidth=1)
axes[0].set_title('Clean EEG', fontsize=11)
axes[0].set_ylabel('uV')

axes[1].plot(t, ecg_artifact, color='#27ae60', linewidth=1.5)
axes[1].set_title('ECG Artifact (alone) -- ~75 bpm', fontsize=11, color='#27ae60')
axes[1].set_ylabel('uV')

axes[2].plot(t, contaminated_ecg, color='#1e8449', linewidth=1)
axes[2].set_title('Contaminated EEG (clean + ECG)', fontsize=11, color='#1e8449')
axes[2].set_ylabel('uV')
axes[2].set_xlabel('Time (s)')

fig.suptitle('ECG (Cardiac) Artifact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Exercise 6:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 7: Artifact 5: Electrode Pop

**Source:** Sudden change in electrode-scalp impedance due to loose connection, dried gel, or faulty electrode.

**Key features:** Abrupt, very fast spike on a **single channel only**. Other channels are unaffected.

**How to fix:** Re-apply electrode paste. Check impedance. Replace electrode.

In [ ]:
# Generate electrode pop
def generate_pop(t, pop_time=1.5, amplitude=250):
    """Simulate a single-channel electrode pop."""
    pop = np.zeros(len(t))
    dt = t - pop_time
    # Sharp upward spike
    spike_mask = (dt > 0) & (dt < 0.02)
    pop[spike_mask] = amplitude * np.sin(np.pi * dt[spike_mask] / 0.02)
    # Brief negative recovery
    recovery_mask = (dt >= 0.02) & (dt < 0.06)
    pop[recovery_mask] = -60 * np.exp(-80 * (dt[recovery_mask] - 0.02))
    return pop

pop_artifact = generate_pop(t)
contaminated_pop = clean_eeg + pop_artifact

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, clean_eeg, 'b-', linewidth=1)
axes[0].set_title('Clean EEG', fontsize=11)
axes[0].set_ylabel('uV')

axes[1].plot(t, pop_artifact, color='#2980b9', linewidth=1.5)
axes[1].set_title('Electrode Pop (alone)', fontsize=11, color='#2980b9')
axes[1].set_ylabel('uV')

axes[2].plot(t, contaminated_pop, color='#1a5276', linewidth=1)
axes[2].set_title('Contaminated EEG (clean + electrode pop)', fontsize=11, color='#1a5276')
axes[2].set_ylabel('uV')
axes[2].set_xlabel('Time (s)')

fig.suptitle('Electrode Pop Artifact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Exercise 7:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 8: Artifact Gallery: All Five Side by Side

Here are all five common EEG artifacts displayed together for easy visual comparison. Study the differences in shape, frequency content, and timing.

In [ ]:
# Gallery: all 5 artifacts
artifacts = [
    ('Eye Blink', blink_artifact, '#e74c3c'),
    ('Muscle (EMG)', muscle_artifact, '#e67e22'),
    ('60 Hz Noise', line_noise, '#8e44ad'),
    ('ECG', ecg_artifact, '#27ae60'),
    ('Electrode Pop', pop_artifact, '#2980b9'),
]

fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
fig.suptitle('EEG Artifact Gallery', fontsize=16, fontweight='bold')

for ax, (name, artifact, color) in zip(axes, artifacts):
    ax.plot(t, artifact, color=color, linewidth=1.2)
    ax.set_ylabel('uV', fontsize=9)
    ax.set_title(name, fontsize=11, color=color, fontweight='bold', loc='left')

axes[-1].set_xlabel('Time (s)', fontsize=12)
plt.tight_layout()
plt.show()

**Exercise 8:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 9: Removing Artifacts with Filters

Signal processing can remove some artifacts while preserving the brain signal:

- **Notch filter** at 60 Hz removes power line noise
- **High-pass filter** (e.g., >1 Hz) removes slow drift and some blink artifact
- **Low-pass filter** (e.g., <40 Hz) removes muscle artifact

**Caution:** Every filter also removes some real brain signal. It is always a tradeoff.

In [ ]:
# Demonstrate artifact removal with filters
# Create a signal contaminated with both blink and 60 Hz noise
dirty = clean_eeg + blink_artifact + line_noise

# Design filters
# 1. Notch filter at 60 Hz
b_notch, a_notch = signal.iirnotch(60, Q=30, fs=fs)
after_notch = signal.filtfilt(b_notch, a_notch, dirty)

# 2. High-pass at 1 Hz to remove blink (slow deflection)
sos_hp = signal.butter(4, 1, btype='highpass', fs=fs, output='sos')
after_hp = signal.sosfiltfilt(sos_hp, after_notch)

# Plot before/after
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Step-by-Step Artifact Removal', fontsize=14, fontweight='bold')

axes[0].plot(t, dirty, color='#c0392b', linewidth=1)
axes[0].set_title('1. Dirty Signal (blink + 60 Hz noise)', fontsize=11, color='#c0392b')
axes[0].set_ylabel('uV')

axes[1].plot(t, after_notch, color='#e67e22', linewidth=1)
axes[1].set_title('2. After 60 Hz Notch Filter', fontsize=11, color='#e67e22')
axes[1].set_ylabel('uV')

axes[2].plot(t, after_hp, color='#27ae60', linewidth=1)
axes[2].set_title('3. After High-Pass Filter (>1 Hz, removes blink)', fontsize=11, color='#27ae60')
axes[2].set_ylabel('uV')

axes[3].plot(t, clean_eeg, 'b-', linewidth=1, alpha=0.5, label='Original clean')
axes[3].plot(t, after_hp, color='#27ae60', linewidth=1, alpha=0.7, label='Recovered')
axes[3].set_title('4. Comparison: Original Clean vs Recovered', fontsize=11)
axes[3].set_ylabel('uV')
axes[3].set_xlabel('Time (s)')
axes[3].legend(fontsize=10)

plt.tight_layout()
plt.show()

print('Note: The recovered signal is close to the original, but not perfect.')
print('The high-pass filter also removes some real low-frequency brain activity.')
print('This is the fundamental tradeoff in artifact removal.')

**Exercise 9:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 10: Exercise 1: Identify the Artifact

**Your turn.** A mystery signal is generated below. It contains clean EEG plus ONE of the five artifact types. Look at the time domain and frequency spectrum to identify which artifact is present.

**Exercise 10:** Try it yourself!

In [ ]:
# EXERCISE 1: Identify the artifact
# ==================================
np.random.seed(314)
fs = 500
t_ex = np.arange(0, 3, 1/fs)

# Clean EEG
clean_ex = (20*np.sin(2*np.pi*10*t_ex) + 
            8*np.sin(2*np.pi*6*t_ex) + 
            6*np.random.randn(len(t_ex)))

# Mystery artifact -- which one is it?
mystery = clean_ex + 12*np.sin(2*np.pi*60*t_ex)

# YOUR CODE HERE
# 1. Plot the mystery signal in the time domain
# 2. Compute and plot its power spectrum
# 3. Identify the artifact type

# Hints:
# - Eye blink: large, slow deflection
# - Muscle: high-frequency fuzz
# - 60 Hz: perfect sinusoidal ripple, spike at exactly 60 Hz
# - ECG: periodic sharp waves ~1 Hz
# - Electrode pop: sudden spike, one channel

## Step 11: Exercise 2: Design a Filter Chain

**Your turn.** Design a filter chain that removes blink artifact while preserving alpha rhythm (8-13 Hz). This requires careful filter design -- you need to attenuate low frequencies (where blinks live) without killing the alpha band.

**Hint:** A high-pass filter with cutoff around 2-4 Hz should work. But test it -- plot the alpha band power before and after filtering.

**Exercise 11:** Try it yourself!

In [ ]:
# EXERCISE 2: Filter chain to remove blinks while preserving alpha
# ================================================================
np.random.seed(55)
fs = 500
t_ex = np.arange(0, 4, 1/fs)

# Signal with strong alpha + blink artifacts
alpha_signal = 40 * np.sin(2*np.pi*10*t_ex)
blinks = generate_blink(t_ex, blink_times=[0.8, 1.9, 3.1], amplitude=150)
dirty_signal = alpha_signal + blinks + 5*np.random.randn(len(t_ex))

# YOUR CODE HERE
# 1. Design a high-pass filter (experiment with cutoff: 1, 2, 4 Hz)
# 2. Apply it to dirty_signal
# 3. Plot before and after
# 4. Calculate alpha band power before and after filtering
#    (Should be preserved!)

# Hint: Use signal.butter() and signal.sosfiltfilt()
# sos = signal.butter(N, cutoff, btype='highpass', fs=fs, output='sos')
# filtered = signal.sosfiltfilt(sos, dirty_signal)

## Step 12: Summary: The Artifact Checklist

**Before calling something abnormal, ask:**

1. Is it in only **one electrode**? Probably electrode pop or bad contact.
2. Is it only in **frontal channels**? Could be an eye blink.
3. Is it **perfectly periodic**? Check if it matches heart rate (ECG) or 60 Hz.
4. Is it **broadband and fuzzy**? Likely muscle.
5. Does it correlate with **patient behavior**? Ask the tech.

**Artifact rejection is not optional.** If you analyze contaminated EEG, your results are wrong. Garbage in, garbage out. Prevention is best, then rejection, then correction.

In [ ]:
# Summary table
print('EEG Artifact Quick Reference')
print('=' * 75)
print(f"{'Artifact':<16} {'Key Feature':<28} {'Fix'}")
print('-' * 75)
summary = [
    ('Eye Blink', 'Large, slow, frontal',       'High-pass filter or ICA'),
    ('Muscle (EMG)', 'High-freq fuzz, temporal', 'Low-pass filter or ICA'),
    ('60 Hz Noise', 'Perfect 60 Hz sinusoid',    'Notch filter at 60 Hz'),
    ('ECG', 'Periodic at heart rate',             'ICA or ECG subtraction'),
    ('Electrode Pop', 'Sharp spike, 1 channel',  'Re-apply paste, reject epoch'),
]
for name, feat, fix in summary:
    print(f"{name:<16} {feat:<28} {fix}")

**Exercise 12:** Try it yourself!

In [ ]:
# YOUR CODE HERE
